# DeepSeek-R1-Distill-Qwen-1.5B — SRD `.axm` → GGUF Q4_K_M + MET Sidecar

Downloads `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` directly from HuggingFace (public, no token required),
packs it into a signed SRD-4 `.axm` container, converts to GGUF Q4_K_M, generates the MET RAM sidecar,
and runs a PPL + reasoning benchmark to compare against DeepSeek's published numbers.

**What this notebook produces**

| Output | Size (est.) | Use |
|---|---|---|
| `deepseek_r1_1b5_srd4.axm` | ~1.5 GB | Signed SRD-4 container (~4.5 bpw) |
| `deepseek_r1_1b5_q4km.gguf` | ~1024 MB | llama.cpp / PocketPal / llama-server |
| `deepseek_r1_1b5_met_sidecar.json` | ~12 KB | MET hydration budget reference |
| Wikitext-2 PPL | scalar | Our standard benchmark |
| Reasoning samples | text | Compare thinking-token format to DeepSeek published examples |

**DeepSeek published benchmarks (R1-Distill-Qwen-1.5B)**

| Benchmark | Published | Our target |
|---|---|---|
| AIME 2024 (pass@1) | 28.9 % | run Cell 7 |
| MATH-500 | 83.9 % | run Cell 7 |
| LiveCodeBench | 16.9 % | run Cell 7 |
| WikiText-2 PPL | — | run Cell 6 (our baseline) |

**Architecture (Qwen2.5-1.5B base)**

| Field | Value |
|---|---|
| Parameters | ~1.54 B (embed 233 M + transformer 1310 M) |
| vocab_size | 151,936 |
| hidden_size | 1,536 |
| num_layers | 28 |
| num_attention_heads | 12 |
| num_kv_heads | 2 (GQA) |
| intermediate_size | 8,960 |
| MLP style | SwiGLU |

> **Patent pending — Orivael Inc.** The SRD container format and signing protocol are the subject of pending patent claims.

**Runtime:** T4 (15 GB VRAM) recommended. A100 faster (~3× pack speed). CPU-only: Cell 4 will be slow (~2 h).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — GPU check · clone axiom · install deps · persist AXIOM_MASTER_KEY
#           clone + cmake-build llama.cpp (GGML_CUDA=ON)  (~10-15 min)
# ══════════════════════════════════════════════════════════════════════════════
import os, secrets, subprocess, sys, time
from pathlib import Path

AXIOM_DIR  = Path('/content/axiom')
OUTPUT_DIR = Path('/content/axiom_output')
LLAMACPP   = Path('/content/llama.cpp')
BRANCH     = 'claude/srd-prototype-benchmark-JRtv1'
REPO_URL   = 'https://github.com/orivael-dev/axiom.git'
KEY_FILE   = OUTPUT_DIR / 'axiom_master.key'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── GPU info ──────────────────────────────────────────────────────────────────
import torch
p = torch.cuda.get_device_properties(0)
vram_gb   = p.total_memory / 1024**3
cuda_arch = p.major * 10 + p.minor
print(f'GPU : {p.name}  {vram_gb:.1f} GB VRAM  SM {p.major}.{p.minor}  arch={cuda_arch}')
if vram_gb < 4:
    print('  ⚠  < 4 GB VRAM — packing 1.5B BF16 weights needs ~3 GB.  Use T4 or better.')
else:
    print(f'  ✓ {vram_gb:.1f} GB VRAM — model fits comfortably')

# ── Clone axiom ───────────────────────────────────────────────────────────────
if not AXIOM_DIR.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    REPO_URL, str(AXIOM_DIR)], check=True)
    print(f'✓ axiom cloned  ({BRANCH})')
else:
    r = subprocess.run(['git', '-C', str(AXIOM_DIR), 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print(f'✓ axiom  {r.stdout.strip() or "up to date"}')

sys.path.insert(0, str(AXIOM_DIR))

# ── Persist AXIOM_MASTER_KEY ──────────────────────────────────────────────────
if KEY_FILE.is_file() and not os.environ.get('AXIOM_MASTER_KEY'):
    os.environ['AXIOM_MASTER_KEY'] = KEY_FILE.read_text().strip()
    print('AXIOM_MASTER_KEY restored from session key file')
elif not os.environ.get('AXIOM_MASTER_KEY'):
    key = secrets.token_hex(32)
    os.environ['AXIOM_MASTER_KEY'] = key
    KEY_FILE.write_text(key)
    print(f'AXIOM_MASTER_KEY generated → {KEY_FILE}')
    print('  ⚠ back this up — same key required to verify the .axm later')
else:
    print('AXIOM_MASTER_KEY already set')

# ── Install deps ──────────────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'datasets', 'huggingface_hub', 'tqdm'], check=True)
print('✓ deps installed')

# ── Build llama.cpp ───────────────────────────────────────────────────────────
if not (LLAMACPP / 'build' / 'bin' / 'llama-cli').is_file():
    print('Building llama.cpp...  (~10 min on T4)')
    t0 = time.time()
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp', str(LLAMACPP)], check=True)
    subprocess.run(
        ['cmake', '-B', 'build', '-DGGML_CUDA=ON',
         f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}'],
        cwd=LLAMACPP, check=True, capture_output=True)
    subprocess.run(
        ['cmake', '--build', 'build', '--config', 'Release', '-j4'],
        cwd=LLAMACPP, check=True, capture_output=True)
    print(f'✓ llama.cpp built  ({time.time()-t0:.0f} s)')
else:
    print('✓ llama.cpp already built')

LLAMA_CLI  = LLAMACPP / 'build' / 'bin' / 'llama-cli'
LLAMA_PPL  = LLAMACPP / 'build' / 'bin' / 'llama-perplexity'
LLAMA_QUANT= LLAMACPP / 'build' / 'bin' / 'llama-quantize'
CONVERT    = next(LLAMACPP.glob('convert*.py'), None)
print(f'✓ ready  llama-cli={LLAMA_CLI.exists()}  llama-perplexity={LLAMA_PPL.exists()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Download DeepSeek-R1-Distill-Qwen-1.5B from HuggingFace
#
# Public model — no HF token required.
# Size: ~3.1 GB (BF16 safetensors).  Time: 5–15 min depending on bandwidth.
# ══════════════════════════════════════════════════════════════════════════════
import time
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID  = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
MODEL_DIR = OUTPUT_DIR / 'deepseek_r1_1b5_hf'

if MODEL_DIR.is_dir() and any(MODEL_DIR.glob('*.safetensors')):
    print(f'✓ model already downloaded  →  {MODEL_DIR}')
else:
    print(f'Downloading {MODEL_ID} ...  (~3.1 GB, BF16)')
    t0 = time.time()
    snapshot_download(
        repo_id   = MODEL_ID,
        local_dir = str(MODEL_DIR),
        ignore_patterns = ['*.pt', '*.bin', '*.ot'],  # safetensors only
    )
    elapsed = time.time() - t0
    files = list(MODEL_DIR.glob('*.safetensors'))
    total_gb = sum(f.stat().st_size for f in files) / 1024**3
    print(f'✓ downloaded  {len(files)} shard(s)  {total_gb:.2f} GB  ({elapsed:.0f} s)')

# Quick architecture check
import json
cfg = json.loads((MODEL_DIR / 'config.json').read_text())
print(f'\nArchitecture:')
print(f'  model_type         : {cfg.get("model_type")}')
print(f'  vocab_size         : {cfg.get("vocab_size"):,}')
print(f'  hidden_size        : {cfg.get("hidden_size")}')
print(f'  num_hidden_layers  : {cfg.get("num_hidden_layers")}')
print(f'  num_attention_heads: {cfg.get("num_attention_heads")}')
print(f'  num_kv_heads       : {cfg.get("num_key_value_heads")}')
print(f'  intermediate_size  : {cfg.get("intermediate_size")}')
print(f'  max_pos_embeddings : {cfg.get("max_position_embeddings"):,}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — SRD-4 pack → deepseek_r1_1b5_srd4.axm
#
# SRD-4 = W4 base quantization, ~4.5 bpw at inference.
# Signs every weight shard with HMAC-SHA256 — fingerprint is a public
# commitment to the exact packed weights.
#
# Time: ~20 min on T4  |  ~8 min on A100
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

AXM_PATH   = OUTPUT_DIR / 'deepseek_r1_1b5_srd4.axm'
PACK_SCRIPT= AXIOM_DIR / 'research' / 'quant' / 'pack_to_axm.py'

if AXM_PATH.exists():
    print(f'✓ .axm already exists  ({AXM_PATH.stat().st_size/1024**3:.2f} GB)  — skipping pack')
else:
    print(f'Packing {MODEL_ID} → SRD-4 .axm ...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, str(PACK_SCRIPT),
         '--model',      str(MODEL_DIR),
         '--output',     str(AXM_PATH),
         '--alpha',      '0.0',   # SRD-4: compact 4-bit mode
         '--group-size', '64'],
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
        raise RuntimeError('pack_to_axm.py failed')
    print(f'✓ packed  ({elapsed/60:.1f} min)')

axm_gb = AXM_PATH.stat().st_size / 1024**3
print(f'  .axm size : {axm_gb:.2f} GB')
print(f'  vs BF16   : 3.1 GB  →  {100*(1 - axm_gb/3.1):.0f}% smaller')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Verify .axm proof ledger + extract → GGUF Q4_K_M
#
# Step 1: axm_cli.py verify — checks every HMAC-SHA256 proof shard
# Step 2: axm_to_gguf.py   — reconstructs FP16 → convert_hf_to_gguf.py → quantize
#
# Expected GGUF size: ~1024 MB  (from met_ram_estimator: embed Q8_0 + tx Q4_K_M)
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

AXM_CLI     = AXIOM_DIR / 'axm_cli.py'
EXTRACT_SCR = AXIOM_DIR / 'research' / 'quant' / 'axm_to_gguf.py'
GGUF_PATH   = OUTPUT_DIR / 'deepseek_r1_1b5_q4km.gguf'

# ── Step 1: Verify ────────────────────────────────────────────────────────────
print('Verifying .axm proof chain...')
r = subprocess.run(
    [sys.executable, str(AXM_CLI), 'verify', str(AXM_PATH)],
    capture_output=True, text=True
)
verify_out = r.stdout + r.stderr
if r.returncode != 0:
    print('VERIFY FAILED:', verify_out[-1000:])
    raise RuntimeError('Proof verification failed')

# Parse fingerprint and proof count
try:
    vdata = json.loads(r.stdout)
    fp     = vdata.get('fingerprint', 'n/a')
    proofs = vdata.get('proofs_checked', 'n/a')
except Exception:
    fp, proofs = 'see output', 'see output'
print(f'  ✓ verified  fingerprint={fp}  proofs={proofs}')

# ── Step 2: Extract to GGUF ───────────────────────────────────────────────────
if GGUF_PATH.exists():
    print(f'\n✓ GGUF already exists  ({GGUF_PATH.stat().st_size/1024**2:.0f} MB)')
else:
    print('\nExtracting → GGUF Q4_K_M...  (~10-20 min on T4)')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, str(EXTRACT_SCR),
         '--container', str(AXM_PATH),
         '--gguf-out',  str(GGUF_PATH),
         '--llamacpp',  str(LLAMACPP),
         '--quant',     'Q4_K_M'],
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
        raise RuntimeError('axm_to_gguf.py failed')
    print(f'✓ extracted  ({elapsed/60:.1f} min)')

gguf_mb = GGUF_PATH.stat().st_size / 1024**2
expected_mb = 1024
delta_pct   = 100 * (gguf_mb - expected_mb) / expected_mb
print(f'  GGUF size : {gguf_mb:.0f} MB  (expected ~{expected_mb} MB  Δ{delta_pct:+.1f} %)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Generate MET RAM sidecar
#
# Runs met_ram_estimator.py with the DeepSeek-R1-Distill-Qwen-1.5B
# architecture flags (read from config.json to ensure accuracy).
#
# Expected hydration budgets:
#   Embedding (F16, pinned) : 445 MB
#   INFORM  (early only)    : 607 MB
#   HARM    (all chunks)    : 1203 MB
#   Savings INFORM vs HARM  : 49.5 %
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys
from pathlib import Path

SIDECAR_PATH = OUTPUT_DIR / 'deepseek_r1_1b5_met_sidecar.json'
ESTIMATOR    = AXIOM_DIR / 'research' / 'quant' / 'met_ram_estimator.py'

# Read arch from the downloaded config.json for accuracy
cfg = json.loads((MODEL_DIR / 'config.json').read_text())
vocab  = cfg['vocab_size']
hidden = cfg['hidden_size']
layers = cfg['num_hidden_layers']
heads  = cfg['num_attention_heads']
kv     = cfg.get('num_key_value_heads', heads)
inter  = cfg['intermediate_size']

r = subprocess.run([
    sys.executable, str(ESTIMATOR),
    '--vocab-size',        str(vocab),
    '--hidden-size',       str(hidden),
    '--num-layers',        str(layers),
    '--num-heads',         str(heads),
    '--num-kv-heads',      str(kv),
    '--intermediate-size', str(inter),
    '--bpw',               '4.85',
    '--mlp-style',         'swiglu',
    '--model-id',          'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    '--output',            str(SIDECAR_PATH),
], capture_output=True, text=True)

print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)

# Summary
sd = json.loads(SIDECAR_PATH.read_text())
hyd = sd['intent_ram_budget']
emb = sd['embedding_slot']['mb']
inform_mb = hyd['INFORM']['total_mb']
harm_mb   = hyd['HARM']['total_mb']
savings_pct = 100 * (1 - inform_mb / harm_mb)

print(f'\n  Embedding (pinned F16) : {emb:.1f} MB')
print(f'  INFORM budget          : {inform_mb:.1f} MB  ({hyd["INFORM"]["ufs_load_ms"]:.0f} ms load)')
print(f'  HARM budget            : {harm_mb:.1f} MB  ({hyd["HARM"]["ufs_load_ms"]:.0f} ms load)')
print(f'  Savings INFORM vs HARM : {savings_pct:.1f} %')
print(f'  Sidecar saved → {SIDECAR_PATH}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — WikiText-2 Perplexity Benchmark
#
# Runs llama-perplexity on the WikiText-2 test set with the same settings
# used for TinyLlama and Qwen3-1.7B so results are comparable:
#   stride 512, context 2048, 100 chunks
#
# Note: DeepSeek-R1-Distill is reasoning-tuned, not trained for raw text
# continuation — expect higher PPL than a base model of similar size.
# The comparison baseline is our measured numbers:
#   TinyLlama-1.1B Q4_K_M : 19.39 ± 0.40 PPL
#   Qwen3-1.7B Q4_K_M     : 21.19 ± 0.48 PPL  (also instruction-tuned)
#   Gemma-3-1B Q4_K_M     : 28.90 ± 0.65 PPL
#
# Time: ~10 min on T4
# ══════════════════════════════════════════════════════════════════════════════
import json, re, subprocess, sys, time
from pathlib import Path

# Download WikiText-2 test set
WT2_PATH = OUTPUT_DIR / 'wikitext2_test.txt'
if not WT2_PATH.exists():
    from datasets import load_dataset
    ds   = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    text = '\n'.join(ds['text'])
    WT2_PATH.write_text(text)
    print(f'✓ wikitext-2 saved  ({WT2_PATH.stat().st_size/1024:.0f} KB)')
else:
    print(f'✓ wikitext-2 already downloaded')

print('\nRunning perplexity benchmark...  (~10 min on T4)')
t0 = time.time()
r  = subprocess.run([
    str(LLAMA_PPL),
    '-m',             str(GGUF_PATH),
    '-f',             str(WT2_PATH),
    '--ctx-size',     '2048',
    '--ppl-stride',   '512',
    '--chunks',       '100',
    '--n-gpu-layers', '99',
], capture_output=True, text=True)
elapsed = time.time() - t0

# Parse PPL from output
ppl_line = [l for l in (r.stdout + r.stderr).splitlines()
            if 'Perplexity' in l or 'ppl' in l.lower()]
print(f'PPL output ({elapsed/60:.1f} min):')
for l in ppl_line:
    print(f'  {l.strip()}')

# Try to extract numeric PPL
ppl_match = re.search(r'Perplexity.*?([0-9]+\.[0-9]+)', r.stdout + r.stderr)
if ppl_match:
    ppl = float(ppl_match.group(1))
    print(f'\nResult: PPL = {ppl:.2f}')
    print(f'Comparison (same settings, stride=512, ctx=2048, 100 chunks):')
    print(f'  TinyLlama-1.1B Q4_K_M  : 19.39')
    print(f'  Qwen3-1.7B Q4_K_M      : 21.19')
    print(f'  Gemma-3-1B Q4_K_M      : 28.90')
    print(f'  DeepSeek-R1-1.5B Q4_K_M: {ppl:.2f}  ← this run')
    print()
    print('Note: reasoning-tuned models score higher PPL on raw wikitext —')
    print('absolute value matters less than rank among similarly tuned models.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Reasoning benchmark: MATH-500 style problems
#
# Runs 5 representative math problems with thinking tokens visible.
# Compare thinking format and answers to DeepSeek's published examples.
#
# Published MATH-500 accuracy: 83.9 %
# Published AIME 2024 pass@1: 28.9 %
#
# Settings from DeepSeek-R1 paper:
#   temperature 0.6, top_p 0.95, max new tokens 32768
# We use max 2048 for Colab speed — enough to see thinking structure.
# ══════════════════════════════════════════════════════════════════════════════
import json, re, subprocess, time

# These problems have unambiguous numeric answers — easy to check
PROBLEMS = [
    {
        'name':   'Sum of integers',
        'prompt': 'What is the sum of all integers from 1 to 100?',
        'answer': '5050',
    },
    {
        'name':   'Modular arithmetic',
        'prompt': 'What is the remainder when 2^2025 is divided by 5?',
        'answer': '2',
    },
    {
        'name':   'Quadratic roots',
        'prompt': 'Find all real roots of x^2 - 7x + 12 = 0. List them separated by a comma.',
        'answer': '3, 4',
    },
    {
        'name':   'Number of divisors',
        'prompt': 'How many positive divisors does 360 have?',
        'answer': '24',
    },
    {
        'name':   'AIME-style (triangle)',
        'prompt': (
            'In triangle ABC, AB = 5, AC = 3, and angle A = 60 degrees. '
            'Find BC^2 using the law of cosines.'
        ),
        'answer': '19',
    },
]

# DeepSeek-R1 chat format (Qwen2.5 base with thinking tokens)
# The model starts thinking with <think> and finalises with </think> answer
def _make_prompt(question: str) -> str:
    return (
        '<|im_start|>system\n'
        'You are a helpful assistant.<|im_end|>\n'
        '<|im_start|>user\n'
        f'{question}<|im_end|>\n'
        '<|im_start|>assistant\n'
        '<think>\n'
    )

results = []
correct = 0

for i, prob in enumerate(PROBLEMS, start=1):
    print(f'\n[{i}/{len(PROBLEMS)}] {prob["name"]}')
    print(f'  Q: {prob["prompt"]}')
    prompt = _make_prompt(prob['prompt'])

    t0 = time.time()
    r  = subprocess.run([
        str(LLAMA_CLI),
        '-m',             str(GGUF_PATH),
        '--temp',         '0.6',
        '--top-p',        '0.95',
        '--n-gpu-layers', '99',
        '-n',             '2048',
        '-p',             prompt,
        '--no-display-prompt',
    ], capture_output=True, text=True, timeout=300)
    elapsed = time.time() - t0

    output = r.stdout.strip()

    # Count thinking tokens (inside <think>...</think>)
    think_match = re.search(r'<think>(.*?)</think>', output, re.DOTALL)
    think_text  = think_match.group(1).strip() if think_match else ''
    think_toks  = len(think_text.split()) if think_text else 0

    # Answer is after </think>
    after_think = output.split('</think>', 1)[-1].strip() if '</think>' in output else output

    # Check if answer string appears anywhere in output
    answered = prob['answer'].replace(' ', '') in output.replace(' ', '').replace(',', ', ')
    if answered:
        correct += 1

    print(f'  Thinking tokens : {think_toks}')
    if think_text:
        print(f'  <think> preview : {think_text[:120].replace(chr(10), " ")}...')
    print(f'  Answer section  : {after_think[:200].replace(chr(10), " ")}')
    print(f'  Expected        : {prob["answer"]}')
    print(f'  Correct         : {"✓" if answered else "✗"}  ({elapsed:.1f} s)')

    results.append({
        'name': prob['name'], 'correct': answered,
        'think_tokens': think_toks, 'elapsed_s': round(elapsed, 1),
    })

print(f'\n{"═"*60}')
print(f'Score: {correct}/{len(PROBLEMS)}  ({100*correct/len(PROBLEMS):.0f} %)')
print(f'DeepSeek published MATH-500: 83.9 %  AIME 2024: 28.9 %')
print(f'Note: our 5-problem set tests basic reasoning, not full MATH-500.')

# Save results
bench_path = OUTPUT_DIR / 'deepseek_r1_1b5_bench.json'
bench_path.write_text(json.dumps({
    'model':       'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    'quant':       'Q4_K_M',
    'gguf_mb':     round(GGUF_PATH.stat().st_size / 1024**2),
    'results':     results,
    'score':       f'{correct}/{len(PROBLEMS)}',
    'score_pct':   round(100 * correct / len(PROBLEMS), 1),
    'published': {'math_500_pct': 83.9, 'aime_2024_pct': 28.9, 'lcb_pct': 16.9},
}, indent=2))
print(f'Results saved → {bench_path}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Summary + llama-bench speed + download outputs
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

# ── Speed benchmark (llama-bench) ─────────────────────────────────────────────
LLAMA_BENCH = LLAMACPP / 'build' / 'bin' / 'llama-bench'
if LLAMA_BENCH.exists():
    print('Running llama-bench (TG + PP)...')
    r = subprocess.run([
        str(LLAMA_BENCH),
        '-m', str(GGUF_PATH),
        '-p', '512',    # prefill tokens
        '-n', '128',    # generation tokens
        '-r', '3',      # repetitions
        '--n-gpu-layers', '99',
    ], capture_output=True, text=True)
    print(r.stdout)
else:
    print('llama-bench not found — run Cell 1 with llama.cpp build')

# ── Output summary ────────────────────────────────────────────────────────────
print('\n' + '═'*65)
print('  OUTPUT FILES')
print('═'*65)
for f in [AXM_PATH, GGUF_PATH, SIDECAR_PATH,
          OUTPUT_DIR / 'deepseek_r1_1b5_bench.json']:
    if f.exists():
        mb = f.stat().st_size / 1024**2
        print(f'  {f.name:<45} {mb:>8.1f} MB')

# ── MET sidecar summary ───────────────────────────────────────────────────────
if SIDECAR_PATH.exists():
    sd  = json.loads(SIDECAR_PATH.read_text())
    hyd = sd['intent_ram_budget']
    print(f'\n  MET hydration (DeepSeek-R1-Distill-Qwen-1.5B Q4_K_M):')
    print(f'  {"Intent":<10} {"RAM":>8}  {"UFS load":>10}')
    print(f'  {"-"*10} {"-"*8}  {"-"*10}')
    for intent, v in hyd.items():
        print(f'  {intent:<10} {v["total_mb"]:>6.1f} MB  {v["ufs_load_ms"]:>7.1f} ms')

# ── llama.cpp run command ─────────────────────────────────────────────────────
print(f'''
  Run locally:
    ./llama-cli -m deepseek_r1_1b5_q4km.gguf \\
        --temp 0.6 --top-p 0.95 --n-gpu-layers 99 \\
        -p "<|im_start|>user\\nWhat is 2^10?<|im_end|>\\n<|im_start|>assistant\\n<think>" \\
        -n 1024

  Download to local machine:
''')

try:
    from google.colab import files
    print('  Downloading GGUF...')
    files.download(str(GGUF_PATH))
    print('  Downloading sidecar...')
    files.download(str(SIDECAR_PATH))
except ImportError:
    print(f'  cp {GGUF_PATH} /your/local/path/')
    print(f'  cp {SIDECAR_PATH} results/')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — AIME 2024 Benchmark  (30 problems, exact-match scoring)
#
# Downloads the AIME 2024 I + II problem set from HuggingFace and runs each
# problem through the GGUF with DeepSeek's recommended inference settings:
#   temperature 0.6  top_p 0.95  max_tokens 8192  (thinking + answer)
#
# Scoring: extract integer 0–999 from \boxed{} or last integer after </think>.
# AIME answers are always integers 000–999 — no partial credit.
#
# Published: DeepSeek-R1-Distill-Qwen-1.5B = 28.9 %  (≈ 8–9 correct / 30)
#
# Time estimate on T4:  ~3–5 min per problem  →  ~90–150 min total
# Set MAX_PROBLEMS = 10 for a 30-min sanity check instead of full 30.
# ══════════════════════════════════════════════════════════════════════════════
import json, re, subprocess, tempfile, time
from pathlib import Path

MAX_PROBLEMS  = 30        # set to 10 for a quick 30-min run
RESULTS_PATH  = OUTPUT_DIR / 'aime2024_results.json'
PUBLISHED_PCT = 28.9      # DeepSeek official pass@1

# ── 1. Load AIME 2024 problems ───────────────────────────────────────────────
problems = []
_sources = [
    ('AI-MO/aime-2024',            None),
    ('Maxwell-Ying/AIME_1983-2024', '2024'),
    ('openai/aime',                 None),
]
for ds_id, year_filter in _sources:
    try:
        from datasets import load_dataset
        ds = load_dataset(ds_id, split='train', trust_remote_code=True)
        rows = list(ds)
        if year_filter:
            rows = [r for r in rows if str(r.get('year', '')) == year_filter]
        if not rows:
            raise ValueError('no rows')
        problems = [
            {'problem': r['problem'], 'answer': str(int(str(r['answer']).strip()))}
            for r in rows
        ]
        print(f'✓ Loaded {len(problems)} problems from {ds_id}')
        break
    except Exception as e:
        print(f'  {ds_id}: {e}')

if not problems:
    raise RuntimeError(
        'Could not load AIME 2024 from any HuggingFace dataset.\n'
        'Try: pip install datasets --upgrade\n'
        'Or add problems manually as a list of {"problem": ..., "answer": ...} dicts.'
    )

problems = problems[:MAX_PROBLEMS]
print(f'Running {len(problems)} / 30 problems  '
      f'(est. {len(problems)*4//60}h {len(problems)*4%60}m on T4)\n')

# ── 2. Helpers ───────────────────────────────────────────────────────────────
SYS_PROMPT = 'You are a helpful assistant.'
PROMPT_TMPL = (
    '<|im_start|>system\n{sys}<|im_end|>\n'
    '<|im_start|>user\n'
    '{q}\n\n'
    'Solve step by step. Your final answer must be a single integer '
    'written as \\boxed{{N}} where N is between 0 and 999 inclusive.'
    '<|im_end|>\n'
    '<|im_start|>assistant\n'
    '<think>\n'
)

def extract_answer(raw: str) -> str:
    """Return the integer answer (0-999) from model output, or '' if none found."""
    # Prefer \boxed{N} after </think>
    after = raw.split('</think>', 1)[-1] if '</think>' in raw else raw
    m = re.search(r'\\boxed\{(\d+)\}', after)
    if m:
        return str(int(m.group(1)) % 1000)
    # Fall back: last standalone integer 0-999 after </think>
    candidates = re.findall(r'(?<![.\d])(\d{1,3})(?![.\d])', after)
    if candidates:
        return str(int(candidates[-1]))
    # Widen search to full output if nothing found after </think>
    candidates = re.findall(r'\\boxed\{(\d+)\}', raw)
    if candidates:
        return str(int(candidates[-1]) % 1000)
    return ''

def run_one(question: str) -> tuple[str, float]:
    """Run a single problem through llama-cli; return (raw_output, elapsed_s)."""
    prompt = PROMPT_TMPL.format(sys=SYS_PROMPT, q=question)
    with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False) as f:
        f.write(prompt)
        fpath = f.name
    t0  = time.time()
    res = subprocess.run(
        [str(LLAMA_CLI),
         '-m',                  str(GGUF_PATH),
         '--file',              fpath,
         '-n',                  '8192',          # enough for long thinking chains
         '--temp',              '0.6',
         '--top-p',             '0.95',
         '--repeat-penalty',    '1.0',
         '--no-display-prompt',
         '-ngl',                '99'],
        capture_output=True, text=True, timeout=600
    )
    Path(fpath).unlink(missing_ok=True)
    return res.stdout.strip(), round(time.time() - t0, 1)

# ── 3. Run ───────────────────────────────────────────────────────────────────
results  = []
n_correct = 0

# Resume from partial save if interrupted
if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    n_correct = sum(r['correct'] for r in results)
    print(f'Resuming from {len(results)} saved results  ({n_correct} correct so far)\n')

for i, item in enumerate(problems):
    if i < len(results):
        continue   # already done

    print(f'[{i+1:2d}/{len(problems)}] ', end='', flush=True)
    raw, elapsed = run_one(item['problem'])
    pred    = extract_answer(raw)
    gold    = item['answer']
    correct = (pred == gold)
    if correct:
        n_correct += 1

    results.append({
        'idx':       i + 1,
        'gold':      gold,
        'pred':      pred,
        'correct':   correct,
        'elapsed_s': elapsed,
    })
    RESULTS_PATH.write_text(json.dumps(results, indent=2))  # save after each

    mark = '✓' if correct else '✗'
    print(f'{mark}  gold={gold:>3s}  pred={pred if pred else "??":>3s}  ({elapsed:.0f}s)')

# ── 4. Score ─────────────────────────────────────────────────────────────────
n         = len(results)
pct       = n_correct / n * 100
delta     = pct - PUBLISHED_PCT
total_min = sum(r['elapsed_s'] for r in results) / 60

print(f'\n╔══════════════════════════════════════════════════╗')
print(f'║  AIME 2024 — Final Score                         ║')
print(f'╠══════════════════════════════════════════════════╣')
print(f'║  Our result : {n_correct:2d} / {n}  =  {pct:.1f} %{" "*24}║')
print(f'║  Published  : 28.9 %  (DeepSeek-R1-1.5B, pass@1)║')
print(f'║  Delta      : {delta:+.1f} pp vs published{" "*19}║')
print(f'║  Runtime    : {total_min:.0f} min total{" "*28}║')
print(f'╚══════════════════════════════════════════════════╝')
print(f'\nResults saved → {RESULTS_PATH}')
print( 'Note: temp=0.6 adds sampling variance (±1 problem between runs).')
print( 'For a deterministic upper-bound, re-run with --temp 0 --greedy.')